In [27]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
from pathlib import Path
from datetime import date, timedelta
import math
import random
import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

PROJECT_PATH = Path("/content/drive/MyDrive/multimodal-fashion-recsys")
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
CHECKPOINTS_PATH = PROJECT_PATH / "checkpoints"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)

Device: cuda


In [29]:
train = pl.scan_parquet(PROCESSED_PATH / "train.parquet")
validation_ground_truth = pl.read_parquet(PROCESSED_PATH / "validation_ground_truth.parquet")
article_mapping = pl.read_parquet(PROCESSED_PATH / "article_mapping.parquet")

NUM_ITEMS = article_mapping.height + 1

print("Items:", NUM_ITEMS - 1)

Items: 105542


In [30]:
sequences = (
    train
    .sort(["customer_idx", "t_dat"])
    .group_by("customer_idx", maintain_order=True)
    .agg(pl.col("article_idx").alias("sequence"))
    .filter(pl.col("sequence").list.len() >= 2)
    .collect()
)

print("Users with sequences:", sequences.height)

Users with sequences: 1221097


In [31]:
sequences.select(
    pl.col("sequence").list.len().min().alias("min"),
    pl.col("sequence").list.len().median().alias("median"),
    pl.col("sequence").list.len().quantile(0.90).alias("q90"),
    pl.col("sequence").list.len().quantile(0.95).alias("q95"),
    pl.col("sequence").list.len().quantile(0.99).alias("q99"),
    pl.col("sequence").list.len().max().alias("max")
)

min,median,q90,q95,q99,max
u32,f64,f64,f64,f64,u32
2,11.0,63.0,95.0,192.0,1876


In [32]:
MAX_LEN = 64
HIDDEN_DIM = 192
NUM_HEADS = 6
NUM_LAYERS = 3
DROPOUT = 0.2

BATCH_SIZE = 512
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5

EPOCHS = 8
PATIENCE = 2

K = 12
VAL_START = date(2020, 9, 9)

In [33]:
class SASRecDataset(Dataset):
    def __init__(self, sequences, num_items, max_len):
        self.sequences = sequences
        self.num_items = num_items
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = np.asarray(self.sequences[idx][-self.max_len - 1:], dtype=np.int64)

        inputs = sequence[:-1]
        positives = sequence[1:]

        input_items = np.zeros(self.max_len, dtype=np.int64)
        positive_items = np.zeros(self.max_len, dtype=np.int64)
        negative_items = np.zeros(self.max_len, dtype=np.int64)

        input_items[-len(inputs):] = inputs
        positive_items[-len(positives):] = positives

        negatives = np.random.randint(1, self.num_items, size=len(positives))

        invalid = np.isin(negatives, sequence)

        while invalid.any():
            negatives[invalid] = np.random.randint(1, self.num_items, size=invalid.sum())
            invalid = np.isin(negatives, sequence)

        negative_items[-len(negatives):] = negatives

        return (
            torch.from_numpy(input_items),
            torch.from_numpy(positive_items),
            torch.from_numpy(negative_items)
        )

In [34]:
train_sequences = sequences["sequence"].to_list()

dataset = SASRecDataset(train_sequences, NUM_ITEMS, MAX_LEN)

input_items, positive_items, negative_items = dataset[0]

print("Training sequences:", len(dataset))
print("Input shape:", input_items.shape)
print("Positive shape:", positive_items.shape)
print("Negative shape:", negative_items.shape)

Training sequences: 1221097
Input shape: torch.Size([64])
Positive shape: torch.Size([64])
Negative shape: torch.Size([64])


In [35]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


generator = torch.Generator()
generator.manual_seed(SEED)

In [36]:
class SASRec(nn.Module):
    def __init__(
        self,
        num_items,
        max_len,
        hidden_dim,
        num_heads,
        num_layers,
        dropout
    ):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.item_embedding = nn.Embedding(num_items, hidden_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, hidden_dim)
        self.dropout = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=num_layers,
            enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(hidden_dim)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.padding_idx is not None:
                with torch.no_grad():
                    module.weight[module.padding_idx].zero_()

        elif isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, input_items):
        seq_len = input_items.size(1)

        positions = torch.arange(seq_len, device=input_items.device).unsqueeze(0)

        x = self.item_embedding(input_items) * math.sqrt(self.hidden_dim)
        x = x + self.position_embedding(positions)
        x = self.dropout(x)

        padding_mask = input_items == 0
        x = x.masked_fill(padding_mask.unsqueeze(-1), 0.0)

        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=input_items.device, dtype=torch.bool),
            diagonal=1
        )

        x = self.transformer(
            x,
            mask=causal_mask,
            src_key_padding_mask=padding_mask
        )

        x = self.norm(x)
        x = x.masked_fill(padding_mask.unsqueeze(-1), 0.0)

        return x

In [37]:
model = SASRec(
    num_items=NUM_ITEMS,
    max_len=MAX_LEN,
    hidden_dim=HIDDEN_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

num_parameters = sum(p.numel() for p in model.parameters())

print(f"Parameters: {num_parameters / 1e6:.2f}M")

Parameters: 21.61M


In [38]:
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    worker_init_fn=seed_worker,
    generator=generator
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-5
)

scaler = torch.amp.GradScaler("cuda")

In [39]:
val_users = validation_ground_truth.select("customer_idx")

val_history = (
    train
    .join(val_users.lazy(), on="customer_idx", how="semi")
    .sort(["customer_idx", "t_dat"])
    .group_by("customer_idx", maintain_order=True)
    .agg(pl.col("article_idx").tail(MAX_LEN).alias("history"))
    .collect()
)

history_map = dict(zip(
    val_history["customer_idx"].to_list(),
    val_history["history"].to_list()
))

In [40]:
val_user_ids = validation_ground_truth["customer_idx"].to_numpy()

val_sequences = np.zeros((len(val_user_ids), MAX_LEN), dtype=np.int64)
has_history = np.zeros(len(val_user_ids), dtype=bool)

for i, user_id in enumerate(val_user_ids):
    history = history_map.get(int(user_id), [])

    if history:
        history = history[-MAX_LEN:]
        val_sequences[i, -len(history):] = history
        has_history[i] = True

print("Validation users:", len(val_user_ids))
print("With history:", has_history.sum())
print("Without history:", (~has_history).sum())

Validation users: 72019
With history: 66624
Without history: 5395


In [41]:
candidate_items = (
    train
    .select("article_idx")
    .unique()
    .collect()["article_idx"]
    .to_numpy()
)

candidate_items_tensor = torch.from_numpy(candidate_items.astype(np.int64)).to(DEVICE)

print("Candidate items:", len(candidate_items))

Candidate items: 102967


In [42]:
recent_top12 = (
    train
    .filter(pl.col("t_dat") >= VAL_START - timedelta(days=14))
    .group_by("article_idx")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(K)
    .collect()["article_idx"]
    .to_list()
)

In [43]:
def average_precision_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    score = 0.0
    hits = 0
    seen = set()

    for i, item in enumerate(predicted[:k], 1):
        if item in actual and item not in seen:
            hits += 1
            score += hits / i

        seen.add(item)

    return score / min(len(actual), k)


def recall_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    return len(actual.intersection(predicted[:k])) / len(actual)


def ndcg_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    dcg = sum(
        1 / math.log2(i + 2)
        for i, item in enumerate(predicted[:k])
        if item in actual
    )

    idcg = sum(
        1 / math.log2(i + 2)
        for i in range(min(len(actual), k))
    )

    return dcg / idcg

In [44]:
actuals = validation_ground_truth["actual"].to_list()

def evaluate_model(model):
    predictions = []
    model.eval()

    with torch.no_grad():
        item_embeddings = model.item_embedding(candidate_items_tensor)

        for start in tqdm(
            range(0, len(val_user_ids), 512),
            desc="Validation",
            leave=False
        ):
            end = min(start + 512, len(val_user_ids))

            batch_sequences = val_sequences[start:end]
            batch_history = has_history[start:end]

            batch_predictions = [recent_top12.copy() for _ in range(end - start)]

            valid_indices = np.flatnonzero(batch_history)

            if len(valid_indices) > 0:
                sequence_tensor = torch.from_numpy(
                    batch_sequences[valid_indices]
                ).to(DEVICE)

                with torch.amp.autocast("cuda", dtype=torch.float16):
                    hidden = model(sequence_tensor)
                    user_embeddings = hidden[:, -1]
                    scores = user_embeddings @ item_embeddings.T

                top_indices = scores.topk(K, dim=1).indices
                recommended = candidate_items_tensor[top_indices].cpu().tolist()

                for position, recommendation in zip(valid_indices, recommended):
                    batch_predictions[position] = recommendation

            predictions.extend(batch_predictions)

    metrics = {
        "MAP@12": sum(
            average_precision_at_k(a, p)
            for a, p in zip(actuals, predictions)
        ) / len(actuals),

        "Recall@12": sum(
            recall_at_k(a, p)
            for a, p in zip(actuals, predictions)
        ) / len(actuals),

        "NDCG@12": sum(
            ndcg_at_k(a, p)
            for a, p in zip(actuals, predictions)
        ) / len(actuals),

        "Coverage": len({
            item
            for prediction in predictions
            for item in prediction
        }) / (NUM_ITEMS - 1)
    }

    return metrics

In [45]:
CHECKPOINTS_PATH.mkdir(parents=True, exist_ok=True)

best_map = -1.0
epochs_without_improvement = 0
training_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()

    total_loss = 0.0
    total_examples = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for input_items, positive_items, negative_items in pbar:
        input_items = input_items.to(DEVICE, non_blocking=True)
        positive_items = positive_items.to(DEVICE, non_blocking=True)
        negative_items = negative_items.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda", dtype=torch.float16):
            hidden = model(input_items)

            positive_embeddings = model.item_embedding(positive_items)
            negative_embeddings = model.item_embedding(negative_items)

            positive_scores = (hidden * positive_embeddings).sum(dim=-1)
            negative_scores = (hidden * negative_embeddings).sum(dim=-1)

            mask = positive_items != 0

            loss = -(
                F.logsigmoid(positive_scores[mask])
                + F.logsigmoid(-negative_scores[mask])
            ).mean()

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        batch_size = input_items.size(0)

        total_loss += loss.item() * batch_size
        total_examples += batch_size

        pbar.set_postfix(loss=f"{total_loss / total_examples:.4f}")

    train_loss = total_loss / total_examples

    metrics = evaluate_model(model)

    training_history.append({
        "epoch": epoch,
        "loss": train_loss,
        **metrics
    })

    print(
        f"Epoch {epoch}: "
        f"loss={train_loss:.4f}, "
        f"MAP@12={metrics['MAP@12']:.6f}, "
        f"Recall@12={metrics['Recall@12']:.6f}"
    )

    if metrics["MAP@12"] > best_map:
        best_map = metrics["MAP@12"]
        epochs_without_improvement = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "metrics": metrics,
            "max_len": MAX_LEN,
            "hidden_dim": HIDDEN_DIM,
            "num_heads": NUM_HEADS,
            "num_layers": NUM_LAYERS,
            "dropout": DROPOUT
        }, CHECKPOINTS_PATH / "sasrec_best.pt")

        print("Saved best checkpoint")

    else:
        epochs_without_improvement += 1

    scheduler.step()

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping")
        break

Epoch 1/8:   0%|          | 0/2385 [00:00<?, ?it/s]

Validation:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch 1: loss=0.6011, MAP@12=0.006101, Recall@12=0.019236
Saved best checkpoint


Epoch 2/8:   0%|          | 0/2385 [00:00<?, ?it/s]

Validation:   0%|          | 0/141 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch 7: loss=0.3443, MAP@12=0.014604, Recall@12=0.036118
Saved best checkpoint


Epoch 8/8:   0%|          | 0/2385 [00:00<?, ?it/s]

Validation:   0%|          | 0/141 [00:00<?, ?it/s]

Epoch 8: loss=0.3381, MAP@12=0.014935, Recall@12=0.036801
Saved best checkpoint


In [46]:
training_history = pl.DataFrame(training_history)

training_history

epoch,loss,MAP@12,Recall@12,NDCG@12,Coverage
i64,f64,f64,f64,f64,f64
1,0.601061,0.006101,0.019236,0.011442,0.030651
2,0.442057,0.008448,0.024649,0.015085,0.062762
3,0.405597,0.010405,0.028294,0.017973,0.087169
4,0.383794,0.011497,0.030296,0.019356,0.114675
5,0.367384,0.012505,0.033147,0.020947,0.135785
6,0.35402,0.013725,0.034835,0.022545,0.156307
7,0.344277,0.014604,0.036118,0.023668,0.164323
8,0.338126,0.014935,0.036801,0.024128,0.168341
